# 6.3 LangChain/LangGraph Agent 简介

LangChain 是一个用于构建语言模型应用的强大框架，而 **LangGraph** 是其生态中的关键组件，专门用于构建基于状态图（Stateful Graph）的多智能体（Agent）工作流。它允许开发者将复杂的任务编排为可控制、可观测、可中断和可恢复的图结构。

在 LangChain 中，**Agent** 是一个能够根据用户输入动态决定使用哪些工具（Tools）、执行哪些动作的“智能体”。它由以下几个核心部分组成：

- **LLM（大语言模型）**：作为决策引擎，决定下一步做什么。
- **Tool（工具）**：函数或 API 接口，供 Agent 调用以执行具体任务。
- **@tool 装饰器**：LangChain 提供的装饰器，用于将普通 Python 函数转换为 LLM 可调用的“工具”。
- **Orchestrator（编排器）**：如 LangGraph 中的 `StateGraph`，负责管理多个节点（Agent 或函数）之间的状态流转与控制逻辑。

---

## `@tool` 装饰器说明

`@tool` 装饰器用于将一个 Python 函数标记为“可被 Agent 调用的工具”。它可以自动提取函数名、描述和参数信息，供 LLM 理解并选择使用。

### 基本语法：
```python
from langchain_core.tools import tool

@tool
def my_function(input: str) -> str:
    """这是一个工具的描述，会被 LLM 读取"""
    return f"处理结果: {input}"

## 示例 1：Hello World 级别 —— 打招呼工具

这个例子展示如何定义一个最简单的工具，并让 Agent 调用它。

In [3]:
#!pip install langchain_core langchain_openai langchain langgraph numpy

In [8]:
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent  # 新导入
from langchain_core.messages import HumanMessage  # 用于输入消息

# 定义一个简单工具
@tool
def greet_user(name: str) -> str:
    """向用户打招呼"""
    return f"Hello, {name}! 很高兴见到你。"

#openai形式
#llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

# 使用火山云模型（兼容 OpenAI 接口）
llm = ChatOpenAI(
    base_url="https://ark.cn-beijing.volces.com/api/v3",
    api_key="80e68c38-22cb-4f71-9377-0768c4d7fe15",
    model="doubao-seed-1-6-250615",  # 替换为你的模型 ID
    temperature=0.1,
    top_p=0.9,
    max_tokens=1024
)

# 创建代理（简化！）
tools = [greet_user]
agent = create_agent(
    model=llm,  # 直接传入 LLM 实例
    tools=tools,
    system_prompt="你是一个有帮助的助手。使用工具完成请求。"  # 简单字符串提示
)

# 执行（直接 invoke，无需 Executor）
result = agent.invoke({
    "messages": [HumanMessage(content="请向小明问好")]
})

In [9]:
result["messages"][-1].content

'Hello, 小明! 很高兴见到你。'

> ✅ 这是最基础的 Agent 使用方式：LLM 解析用户请求 → 调用 `greet_user` 工具 → 返回结果。

## 示例 2：稍复杂 —— 天气查询 + 建议穿衣 Agent

该示例结合两个工具：获取天气、根据天气建议穿衣。展示 Agent 如何进行多步推理与工具调用。

In [10]:
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent  # 新导入：迁移到 langchain.agents
from langchain_core.messages import HumanMessage  # 用于输入消息

# 工具1：模拟获取天气
@tool
def get_weather(city: str) -> str:
    """获取指定城市的天气（模拟）"""
    weather_data = {"北京": "晴天，25°C", "上海": "下雨，20°C", "广州": "多云，30°C"}
    return weather_data.get(city, "未知城市")

# 工具2：根据天气建议穿衣
@tool
def suggest_clothing(weather_info: str) -> str:
    """根据天气信息建议穿什么衣服"""
    if "下雨" in weather_info:
        return "建议带伞，穿防水外套"
    elif "25" in weather_info or "30" in weather_info:
        return "天气炎热，建议穿短袖和防晒衣"
    else:
        return "天气舒适，穿长袖即可"

# 使用火山云模型（兼容 OpenAI 接口）
llm = ChatOpenAI(
    base_url="https://ark.cn-beijing.volces.com/api/v3",
    api_key="80e68c38-22cb-4f71-9377-0768c4d7fe15",
    model="doubao-seed-1-6-250615",  # 替换为你的模型 ID
    temperature=0.1,
    top_p=0.9,
    max_tokens=1024
)

tools = [get_weather, suggest_clothing]

# 创建代理（使用系统提示指导工具使用）
system_prompt = "你是一个有帮助的助手。使用工具逐步完成用户请求，例如先查询天气，再基于天气给出建议。"
agent_executor = create_agent(
    llm, 
    tools, 
    system_prompt=system_prompt,  # 修复：使用 prompt 参数（字符串提示）
    debug=True  # 可选：打印调试信息，观察工具调用
)

# 执行复合任务
query = "北京今天天气怎么样？我该穿什么衣服？"
result = agent_executor.invoke({
    "messages": [HumanMessage(content=query)]  # 输入作为消息
})

# 输出最终响应（最后一条消息的内容）
print(result["messages"][-1].content)
# 预期输出示例: 北京今天是晴天，25°C。建议穿短袖和防晒衣。

[values] {'messages': [HumanMessage(content='北京今天天气怎么样？我该穿什么衣服？', additional_kwargs={}, response_metadata={}, id='cb88cbcd-a057-437a-9620-ee4283f9400d')]}
[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 497, 'prompt_tokens': 530, 'total_tokens': 1027, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 470, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'doubao-seed-1-6-250615', 'system_fingerprint': None, 'id': '021761044361146e685748fb40c54556720fa0f3907bebcbcedb4', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--940cc02b-c18d-4656-abe7-93f38bfc1d28-0', tool_calls=[{'name': 'get_weather', 'args': {'city': '北京'}, 'id': 'call_mlf8xtsjc8mpf0fojkdgv8rg', 'type': 'tool_call'}], usage_metadata={

> ✅ 该 Agent 自动完成以下流程：
> 1. 调用 `get_weather("北京")` 获取天气；
> 2. 将结果传给 `suggest_clothing`；
> 3. 综合输出最终建议。

## 总结

| 特性 | 说明 |
|------|------|
| `@tool` 装饰器 | 将函数暴露给 LLM，使其可被智能调用 |
| Agent 决策能力 | LLM 根据输入和工具描述，决定调用哪个工具 |
| LangGraph 的作用 | 在更复杂场景中，用图结构管理多个 Agent 和状态流转（如循环、条件分支） |

通过 LangChain 的 `@tool` 和 Agent 机制，我们可以轻松构建具备“感知-思考-行动”能力的智能系统，从简单问答到复杂自动化工作流均可实现。

# 将 AirSim API 封装为 Agent 工具

这是连接 LLM Agent 的抽象世界与 AirSim 模拟器物理世界的关键一步。我们需要将底层的 AirSim API 调用封装成 LLM 能够理解和调用的“工具”函数。

以下是几个核心工具的 Python 实现代码。我们使用了 LangChain/LangGraph 中的 `@tool` 装饰器，它能自动从函数的**函数名、参数、类型提示**和**文档字符串（docstring）**中推断出工具的模式（schema），供 LLM 参考。这些工具是智能体与模拟环境交互的唯一途径。

## 核心概念

- **工具封装目的**：将复杂的物理世界 API 抽象为语言模型可理解的高层函数。
- **@tool 装饰器作用**：
  - 自动生成工具描述（Tool Schema）。
  - 使 LLM 能够根据任务需求选择并调用正确的工具。
  - 实现“思考-行动”循环中的“行动”环节。
- **交互原则**：Agent 通过调用这些工具与 AirSim 环境进行通信，实现感知（如获取图像）和执行（如飞行控制）。

> ✅ 提示：良好的文档字符串（docstring）至关重要，它直接影响 LLM 是否能正确理解工具的用途和参数含义。

In [13]:
import sys
sys.path.append('../external-libraries')

import airsim
from langchain_core.tools import tool
from typing import Dict

# 初始化一个全局的AirSim客户端
client = airsim.MultirotorClient()
client.confirmConnection()

@tool
def takeoff(drone_id: str) -> str:
    """
    命令指定的无人机起飞并悬停。
    :param drone_id: 要控制的无人机的名称 (例如, 'Drone1').
    """
    print(f"Executing: takeoff for {drone_id}")
    # 必须先解锁API控制
    client.enableApiControl(True, vehicle_name=drone_id)
    client.armDisarm(True, vehicle_name=drone_id)
    # 异步起飞并等待完成
    client.takeoffAsync(vehicle_name=drone_id).join()
    return f"无人机 {drone_id} 已经成功起飞。"

@tool
def fly_to_position(drone_id: str, x: float, y: float, z: float) -> str:
    """
    命令指定的无人机以5米/秒的速度飞往一个NED坐标。
    Z坐标是负数表示向上。
    :param drone_id: 要控制的无人机的名称。
    :param x: 目标位置的X坐标。
    :param y: 目标位置的Y坐标。
    :param z: 目标位置的Z坐标 (负数代表高度)。
    """
    print(f"Executing: fly_to_position for {drone_id} to ({x}, {y}, {z})")
    client.moveToPositionAsync(x, y, z, 5, vehicle_name=drone_id).join()
    return f"无人机 {drone_id} 已到达目标位置 ({x}, {y}, {z})。"

@tool
def get_drone_state(drone_id: str) -> Dict:
    """
    获取指定无人机的当前状态，包括位置和姿态。
    :param drone_id: 要查询的无人机的名称。
    """
    print(f"Executing: get_drone_state for {drone_id}")
    state = client.getMultirotorState(vehicle_name=drone_id)
    # 返回一个字典，方便LLM处理
    return {
        "position": {
            "x_val": state.kinematics_estimated.position.x_val,
            "y_val": state.kinematics_estimated.position.y_val,
            "z_val": state.kinematics_estimated.position.z_val,
        },
        "orientation": {
            "w_val": state.kinematics_estimated.orientation.w_val,
            "x_val": state.kinematics_estimated.orientation.x_val,
            "y_val": state.kinematics_estimated.orientation.y_val,
            "z_val": state.kinematics_estimated.orientation.z_val,
        }
    }


Connect error on fd 2928: WSAECONNREFUSED
Connect error on fd 2928: WSAECONNREFUSED
Connect error on fd 2928: WSAECONNREFUSED

KeyboardInterrupt



通过这种方式，我们为LLM提供了一个清晰、结构化的接口。

LLM不需要了解airsim.MultirotorClient的复杂细节，只需根据工具的文档字符串描述来决定何时调用fly_to_position即可。